In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.ensemble import (RandomForestRegressor, RandomForestClassifier)
from sklearn.metrics import (mean_absolute_error,mean_squared_error, r2_score, accuracy_score,classification_report, recall_score, make_scorer)
from sklearn.inspection import permutation_importance
from skopt import BayesSearchCV
from skopt.space import Integer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import make_scorer, recall_score, ConfusionMatrixDisplay
import joblib

In [ ]:
cwd = Path.cwd().resolve()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd

DATA_PATH = BASE_DIR / "data" / "processed" / "modeling_dataset.csv"

OUTPUT_DIR = BASE_DIR / "outputs" / "RF_modeling"
FIGURE_DIR = BASE_DIR / "outputs" / "final_figures" / "RF_modeling"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Data path:", DATA_PATH)

# Loading modelling dataset

fleet_df = pd.read_csv(DATA_PATH)
fleet_df["date"] = pd.to_datetime(fleet_df["date"])
fleet_df = fleet_df.sort_values(["blower_id", "date"]).copy()

print("Dataset loaded successfully.")
print("Shape:", fleet_df.shape)
print("Date range:", fleet_df["date"].min(), "to", fleet_df["date"].max())
print("Number of blowers:", fleet_df["blower_id"].nunique())

fleet_df.head()

In [ ]:
# Checking required columns

required_columns = [
    "date",
    "blower_id",
    "site_id",
    "operational_class",
    "daily_op_hours",
    "cumulative_op_hours",
    "days_since_maintenance",
    "hours_since_maintenance",
    "daily_load_percent",
    "humidity",
    "wind_gust_kph",
    "dust_index",
    "pressure_diff_psi",
    "casing_temperature_c",
    "vibration_mm_s",
    "RUL_days",
    "maintenance_event",
    "maintenance_due_14d",
    "maintenance_due_30d",
    "degradation_index",
    "health_score"
]

missing_columns = [
    col for col in required_columns if col not in fleet_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required modelling columns are available.")

RANDOM FOREST REGRESSOR

In [ ]:
# Defining target and three feature set options
target = "RUL_days"

feature_sets = {
    "Option_A_Full_Model": [
        "daily_op_hours",
        "cumulative_op_hours",
        "days_since_maintenance",
        "hours_since_maintenance",
        "daily_load_percent",
        "humidity",
        "wind_gust_kph",
        "dust_index",
        "pressure_diff_psi",
        "casing_temperature_c",
        "vibration_mm_s",
        "operational_class",
    ],

    "Option_B_Reduced_With_Duty_Class": [
        "daily_op_hours",
        "daily_load_percent",
        "humidity",
        "wind_gust_kph",
        "dust_index",
        "pressure_diff_psi",
        "casing_temperature_c",
        "vibration_mm_s",
        "operational_class",
    ],

    "Option_C_Condition_Only": [
        "daily_op_hours",
        "daily_load_percent",
        "humidity",
        "wind_gust_kph",
        "dust_index",
        "pressure_diff_psi",
        "casing_temperature_c",
        "vibration_mm_s",
    ]
}

for option_name, features in feature_sets.items():
    print("\n" + "=" * 70)
    print(option_name)
    print("=" * 70)
    for feature in features:
        print("-", feature)

In [ ]:
#Defining train-test split

split_date = pd.Timestamp("2025-10-01")

print("Split date:", split_date)

In [ ]:
#Functioning to train and evaluate one Random Forest model option

def train_random_forest_option(option_name, model_features, fleet_df, target, split_date):
    print("\n" + "=" * 80)
    print(f"Training {option_name}")
    print("=" * 80)

    model_df = fleet_df.dropna(subset=model_features + [target]).copy()

    X = model_df[model_features].copy()
    y = model_df[target].copy()

    if "operational_class" in X.columns:
        X = pd.get_dummies(
            X,
            columns=["operational_class"],
            drop_first=False
        )

    train_mask = model_df["date"] < split_date
    test_mask = model_df["date"] >= split_date

    X_train = X.loc[train_mask]
    X_test = X.loc[test_mask]

    y_train = y.loc[train_mask]
    y_test = y.loc[test_mask]

    print("Training records:", X_train.shape[0])
    print("Testing records:", X_test.shape[0])
    print("Number of features:", X_train.shape[1])

    rf_regressor = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )

    rf_regressor.fit(X_train, y_train)

    y_pred = rf_regressor.predict(X_test)
    y_pred = np.clip(y_pred, 0, None)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print("\nResults")
    print("-" * 40)
    print(f"MAE:  {mae:.2f} days")
    print(f"RMSE: {rmse:.2f} days")
    print(f"R²:   {r2:.4f}")

    metrics_df = pd.DataFrame([{
        "option": option_name,
        "model": "Random Forest Regressor",
        "target": target,
        "number_of_features": X_train.shape[1],
        "train_start": model_df.loc[train_mask, "date"].min(),
        "train_end": model_df.loc[train_mask, "date"].max(),
        "test_start": model_df.loc[test_mask, "date"].min(),
        "test_end": model_df.loc[test_mask, "date"].max(),
        "mae_days": mae,
        "rmse_days": rmse,
        "r2_score": r2
    }])

    test_predictions = model_df.loc[test_mask, [
        "date",
        "blower_id",
        "site_id",
        "operational_class",
        "RUL_days",
        "degradation_index",
        "health_score",
        "maintenance_event",
        "maintenance_due_14d",
        "maintenance_due_30d"
    ]].copy()

    pred_col = f"{option_name}_rf_predicted_RUL_days"
    error_col = f"{option_name}_rf_absolute_error_days"
    residual_col = f"{option_name}_rf_residual_days"

    test_predictions[pred_col] = y_pred

    test_predictions[error_col] = (
        test_predictions["RUL_days"] - test_predictions[pred_col]
    ).abs()

    test_predictions[residual_col] = (
        test_predictions["RUL_days"] - test_predictions[pred_col]
    )

    test_predictions[f"{option_name}_rf_maintenance_due_14d_pred"] = (
        test_predictions[pred_col] <= 14
    ).astype(int)

    test_predictions[f"{option_name}_rf_maintenance_due_30d_pred"] = (
        test_predictions[pred_col] <= 30
    ).astype(int)

    importance_df = pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf_regressor.feature_importances_
    }).sort_values(by="importance", ascending=False)

    full_predictions = rf_regressor.predict(X)
    full_predictions = np.clip(full_predictions, 0, None)

    full_prediction_df = model_df.copy()
    full_prediction_df[pred_col] = full_predictions

    full_prediction_df[f"{option_name}_rf_maintenance_due_14d_pred"] = (
        full_prediction_df[pred_col] <= 14
    ).astype(int)

    full_prediction_df[f"{option_name}_rf_maintenance_due_30d_pred"] = (
        full_prediction_df[pred_col] <= 30
    ).astype(int)

    full_prediction_df["data_split"] = np.where(
        full_prediction_df["date"] < split_date,
        "train",
        "test"
    )

    return {
        "option_name": option_name,
        "model": rf_regressor,
        "metrics": metrics_df,
        "test_predictions": test_predictions,
        "feature_importance": importance_df,
        "full_predictions": full_prediction_df,
        "pred_col": pred_col,
        "residual_col": residual_col,
        "X_train_columns": X_train.columns.tolist()
    }

In [ ]:
# Training all three Random Forest options

rf_results = {}

for option_name, features in feature_sets.items():
    rf_results[option_name] = train_random_forest_option(
        option_name=option_name,
        model_features=features,
        fleet_df=fleet_df,
        target=target,
        split_date=split_date
    )

In [ ]:
#Combining and saving metrics comparison

metrics_comparison = pd.concat(
    [result["metrics"] for result in rf_results.values()],
    ignore_index=True
).sort_values(by="rmse_days")

metrics_comparison_path = OUTPUT_DIR / "random_forest_three_option_metrics_comparison.csv"

metrics_comparison.to_csv(
    metrics_comparison_path,
    index=False
)

metrics_comparison

In [ ]:
#Plotting metrics comparison: MAE and RMSE

plot_metrics = metrics_comparison.copy()

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(plot_metrics))
width = 0.35

ax.bar(
    x - width / 2,
    plot_metrics["mae_days"],
    width,
    label="MAE"
)

ax.bar(
    x + width / 2,
    plot_metrics["rmse_days"],
    width,
    label="RMSE"
)

ax.set_title("Random Forest Feature Set Comparison: MAE and RMSE")
ax.set_xlabel("Feature Set Option")
ax.set_ylabel("Error (days)")
ax.set_xticks(x)
ax.set_xticklabels(plot_metrics["option"], rotation=20, ha="right")
ax.legend()
ax.grid(True, axis="y")

plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "random_forest_three_option_mae_rmse_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Plotting metrics comparison: MAE and RMSE

plot_metrics = metrics_comparison.copy()

fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(plot_metrics))
width = 0.35

ax.bar(
    x - width / 2,
    plot_metrics["mae_days"],
    width,
    label="MAE"
)

ax.bar(
    x + width / 2,
    plot_metrics["rmse_days"],
    width,
    label="RMSE"
)

ax.set_title("Random Forest Feature Set Comparison: MAE and RMSE")
ax.set_xlabel("Feature Set Option")
ax.set_ylabel("Error (days)")
ax.set_xticks(x)
ax.set_xticklabels(plot_metrics["option"], rotation=20, ha="right")
ax.legend()
ax.grid(True, axis="y")

plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "random_forest_three_option_mae_rmse_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#Plotting metrics comparison: R²

fig, ax = plt.subplots(figsize=(9, 5))

ax.bar(
    plot_metrics["option"],
    plot_metrics["r2_score"]
)

ax.set_title("Random Forest Feature Set Comparison: R²")
ax.set_xlabel("Feature Set Option")
ax.set_ylabel("R² Score")
ax.set_xticklabels(plot_metrics["option"], rotation=20, ha="right")
ax.grid(True, axis="y")

plt.tight_layout()

fig.savefig(
    FIGURE_DIR / "random_forest_three_option_r2_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Saving outputs and plots for each option

for option_name, result in rf_results.items():

    safe_name = option_name.lower()

    model_path = OUTPUT_DIR / f"{safe_name}_random_forest_rul_model.joblib"
    joblib.dump(result["model"], model_path)

    test_predictions_path = OUTPUT_DIR / f"{safe_name}_test_predictions.csv"
    result["test_predictions"].to_csv(test_predictions_path, index=False)

    importance_path = OUTPUT_DIR / f"{safe_name}_feature_importance.csv"
    result["feature_importance"].to_csv(importance_path, index=False)

    full_predictions_path = PROCESSED_DIR / f"{safe_name}_rf_modeling_dataset_with_predictions.csv"
    result["full_predictions"].to_csv(full_predictions_path, index=False)

    test_policy_path = PROCESSED_DIR / f"{safe_name}_rf_test_policy_predictions.csv"
    result["test_predictions"].to_csv(test_policy_path, index=False)

    print(f"\nSaved outputs for {option_name}")
    print("- Model:", model_path)
    print("- Test predictions:", test_predictions_path)
    print("- Feature importance:", importance_path)
    print("- Full predictions:", full_predictions_path)
    print("- Test-only policy predictions:", test_policy_path)

In [ ]:
# Plotting feature importance for each option

for option_name, result in rf_results.items():

    safe_name = option_name.lower()

    importance_df = result["feature_importance"]

    top_n = min(12, len(importance_df))

    plot_df = importance_df.sort_values(
        by="importance",
        ascending=True
    ).tail(top_n)

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.barh(
        plot_df["feature"],
        plot_df["importance"]
    )

    ax.set_title(f"Random Forest Feature Importance: {option_name}")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    ax.grid(True, axis="x")

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / f"{safe_name}_feature_importance.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# Plotting actual vs predicted RUL scatter for each option

for option_name, result in rf_results.items():

    safe_name = option_name.lower()

    test_predictions = result["test_predictions"]
    pred_col = result["pred_col"]

    fig, ax = plt.subplots(figsize=(7, 6))

    ax.scatter(
        test_predictions["RUL_days"],
        test_predictions[pred_col],
        alpha=0.7
    )

    min_value = min(
        test_predictions["RUL_days"].min(),
        test_predictions[pred_col].min()
    )

    max_value = max(
        test_predictions["RUL_days"].max(),
        test_predictions[pred_col].max()
    )

    ax.plot(
        [min_value, max_value],
        [min_value, max_value],
        linestyle="--"
    )

    ax.set_title(f"Random Forest Actual vs Predicted RUL: {option_name}")
    ax.set_xlabel("Actual RUL (days)")
    ax.set_ylabel("Predicted RUL (days)")
    ax.grid(True)

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / f"{safe_name}_actual_vs_predicted_rul.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
# Plotting residual distribution for each option

for option_name, result in rf_results.items():

    safe_name = option_name.lower()

    test_predictions = result["test_predictions"]
    residual_col = result["residual_col"]

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.hist(
        test_predictions[residual_col],
        bins=30
    )

    ax.set_title(f"Random Forest Residual Distribution: {option_name}")
    ax.set_xlabel("Residual Error (Actual - Predicted RUL)")
    ax.set_ylabel("Frequency")
    ax.grid(True)

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / f"{safe_name}_residual_distribution.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
#Fleet-average actual vs predicted RUL trend for each option

for option_name, result in rf_results.items():

    safe_name = option_name.lower()

    test_predictions = result["test_predictions"]
    pred_col = result["pred_col"]

    fleet_rul_trend = (
        test_predictions
        .groupby("date")[["RUL_days", pred_col]]
        .mean()
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(
        fleet_rul_trend["date"],
        fleet_rul_trend["RUL_days"],
        label="Actual average RUL"
    )

    ax.plot(
        fleet_rul_trend["date"],
        fleet_rul_trend[pred_col],
        linestyle="--",
        label="Predicted average RUL"
    )

    ax.set_title(f"Random Forest Average RUL Trend: {option_name}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Average RUL (days)")
    ax.legend()
    ax.grid(True)

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / f"{safe_name}_average_actual_vs_predicted_rul.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
#Selected blower actual vs predicted RUL trend for each option

for option_name, result in rf_results.items():

    safe_name = option_name.lower()

    test_predictions = result["test_predictions"]
    pred_col = result["pred_col"]

    selected_blowers = test_predictions["blower_id"].unique()[:3]

    fig, ax = plt.subplots(figsize=(11, 5))

    colour_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for i, blower in enumerate(selected_blowers):
        subset = (
            test_predictions[test_predictions["blower_id"] == blower]
            .sort_values("date")
        )

        colour = colour_cycle[i % len(colour_cycle)]

        ax.plot(
            subset["date"],
            subset["RUL_days"],
            color=colour,
            linestyle="-",
            label=f"{blower} actual"
        )

        ax.plot(
            subset["date"],
            subset[pred_col],
            color=colour,
            linestyle="--",
            label=f"{blower} predicted"
        )

    ax.set_title(f"Random Forest Actual vs Predicted RUL for Selected Blowers: {option_name}")
    ax.set_xlabel("Date")
    ax.set_ylabel("RUL (days)")
    ax.legend(fontsize="small", bbox_to_anchor=(1.05, 1), loc="upper left")
    ax.grid(True)

    plt.tight_layout()

    fig.savefig(
        FIGURE_DIR / f"{safe_name}_selected_blowers_actual_vs_predicted_rul.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
#Creating combined test-only policy prediction dataset for RF options

combined_policy_df = None

for option_name, result in rf_results.items():

    test_df = result["test_predictions"].copy()

    keep_cols = [
        "date",
        "blower_id",
        "site_id",
        "operational_class",
        "RUL_days",
        "degradation_index",
        "health_score",
        "maintenance_event",
        "maintenance_due_14d",
        "maintenance_due_30d",
        result["pred_col"],
        f"{option_name}_rf_maintenance_due_14d_pred",
        f"{option_name}_rf_maintenance_due_30d_pred"
    ]

    test_df = test_df[keep_cols].copy()

    if combined_policy_df is None:
        combined_policy_df = test_df
    else:
        merge_cols = [
            "date",
            "blower_id",
            "site_id",
            "operational_class",
            "RUL_days",
            "degradation_index",
            "health_score",
            "maintenance_event",
            "maintenance_due_14d",
            "maintenance_due_30d"
        ]

        combined_policy_df = combined_policy_df.merge(
            test_df,
            on=merge_cols,
            how="left"
        )

combined_policy_path = PROCESSED_DIR / "random_forest_three_option_test_policy_predictions.csv"

combined_policy_df.to_csv(
    combined_policy_path,
    index=False
)

print("Combined policy-ready RF test prediction dataset saved to:")
print(combined_policy_path)

combined_policy_df.head()

RANDOM FOREST CLASSIFIER

In [ ]:
#Creating combined test-only policy prediction dataset for RF options

combined_policy_df = None

for option_name, result in rf_results.items():

    test_df = result["test_predictions"].copy()

    keep_cols = [
        "date",
        "blower_id",
        "site_id",
        "operational_class",
        "RUL_days",
        "degradation_index",
        "health_score",
        "maintenance_event",
        "maintenance_due_14d",
        "maintenance_due_30d",
        result["pred_col"],
        f"{option_name}_rf_maintenance_due_14d_pred",
        f"{option_name}_rf_maintenance_due_30d_pred"
    ]

    test_df = test_df[keep_cols].copy()

    if combined_policy_df is None:
        combined_policy_df = test_df
    else:
        merge_cols = [
            "date",
            "blower_id",
            "site_id",
            "operational_class",
            "RUL_days",
            "degradation_index",
            "health_score",
            "maintenance_event",
            "maintenance_due_14d",
            "maintenance_due_30d"
        ]

        combined_policy_df = combined_policy_df.merge(
            test_df,
            on=merge_cols,
            how="left"
        )

combined_policy_path = PROCESSED_DIR / "random_forest_three_option_test_policy_predictions.csv"

combined_policy_df.to_csv(
    combined_policy_path,
    index=False
)

print("Combined policy-ready RF test prediction dataset saved to:")
print(combined_policy_path)

combined_policy_df.head()

In [ ]:
#Displaying classifier feature importance

print("Feature importance: maintenance_due_14d")
display(classifier_results["maintenance_due_14d"]["feature_importance"])

print("\nFeature importance: maintenance_due_30d")
#Displaying classifier feature importance

print("Feature importance: maintenance_due_14d")
display(classifier_results["maintenance_due_30d"]["feature_importance"])

print("\nFeature importance: maintenance_due_30d")
display(importance_df)

In [ ]:
#Summary

print("Random Forest three-option modelling completed.")

print("\nMetrics comparison:")
display(metrics_comparison)

print("\nSaved outputs:")
print(f"- Metrics comparison: {metrics_comparison_path}")
print(f"- Combined policy-ready RF test predictions: {combined_policy_path}")
print(f"- Output directory: {OUTPUT_DIR}")
print(f"- Figure directory: {FIGURE_DIR}")